# Error Handling with try/except in Python

This notebook covers Python's exception handling mechanism, which helps you gracefully manage errors that occur during program execution.

## Understanding Exceptions in Python

Exceptions are events that occur during the execution of a program that disrupt the normal flow of instructions. When an error occurs in Python, the interpreter creates an exception object. If this exception is not handled properly, the program will terminate with an error message.

Common built-in exceptions in Python include:

In [ ]:
# Let's generate some common exceptions to see what they look like

# 1. ZeroDivisionError
try:
    10 / 0
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")

# 2. TypeError
try:
    "hello" + 5
except TypeError as e:
    print(f"TypeError: {e}")

# 3. IndexError
try:
    my_list = [1, 2, 3]
    print(my_list[10])
except IndexError as e:
    print(f"IndexError: {e}")

# 4. KeyError
try:
    my_dict = {"a": 1, "b": 2}
    print(my_dict["c"])
except KeyError as e:
    print(f"KeyError: {e}")

# 5. FileNotFoundError
try:
    with open("non_existent_file.txt", "r") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

### Exception Hierarchy

Python has a well-defined exception hierarchy. All exceptions inherit from the `BaseException` class. Most exceptions you'll handle inherit from `Exception`.

In [ ]:
# Let's visualize part of the exception hierarchy
import inspect

def print_exception_hierarchy(exception_class, indent=0):
    print(" " * indent + exception_class.__name__)
    for subclass in exception_class.__subclasses__():
        print_exception_hierarchy(subclass, indent + 4)

# Print a portion of the exception hierarchy
print_exception_hierarchy(BaseException)

## Basic try/except Structure

The basic structure of exception handling in Python is the `try`/`except` block:

In [ ]:
# Basic try/except structure
try:
    # Code that might raise an exception
    result = 10 / 0
except:
    # Code that executes if an exception occurs
    print("An error occurred!")

However, using a bare `except` clause without specifying which exception to catch is generally considered bad practice because it catches all exceptions, which can hide bugs and make debugging difficult.

## Handling Specific Exception Types

It's better to catch specific exceptions that you expect might occur. This makes your code more robust and easier to debug:

In [ ]:
# Handling specific exception types
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Error: Division by zero is not allowed"

print(divide(10, 2))  # Works fine
print(divide(10, 0))  # Handles the error gracefully

You can also capture the exception object itself, which can be useful for logging or displaying more detailed error messages:

In [ ]:
def parse_json(json_str):
    try:
        import json
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Invalid JSON: {e}")
        return None

# Valid JSON
result1 = parse_json('{"name": "Alice", "age": 30}')
print("Valid JSON result:", result1)

# Invalid JSON
result2 = parse_json('{name: Alice, age: 30}')
print("Invalid JSON result:", result2)

## Multiple except Blocks

You can handle different exception types differently by using multiple `except` blocks:

In [ ]:
def process_file(filename):
    try:
        with open(filename, 'r') as file:
            content = file.read()
            value = int(content)
            result = 100 / value
            return result
    except FileNotFoundError:
        print(f"Error: The file '{filename}' does not exist.")
        return None
    except ValueError:
        print(f"Error: The file does not contain a valid integer.")
        return None
    except ZeroDivisionError:
        print(f"Error: The file contains zero, which cannot be used as a divisor.")
        return None

# Let's create some test files
import os

# File with a valid number
with open("valid_number.txt", "w") as f:
    f.write("5")

# File with zero
with open("zero.txt", "w") as f:
    f.write("0")

# File with non-numeric content
with open("invalid_number.txt", "w") as f:
    f.write("hello")

# Test the function
print("Processing valid_number.txt:", process_file("valid_number.txt"))
print("Processing zero.txt:", process_file("zero.txt"))
print("Processing invalid_number.txt:", process_file("invalid_number.txt"))
print("Processing non_existent_file.txt:", process_file("non_existent_file.txt"))

### Catching Multiple Exceptions

You can also catch multiple exceptions in a single `except` block if you want to handle them in the same way:

In [ ]:
def safe_operation(a, b):
    try:
        # This could raise TypeError, ZeroDivisionError, or other exceptions
        return a / b
    except (TypeError, ZeroDivisionError) as e:
        # Handle both exceptions the same way
        print(f"Operation failed: {e}")
        return None

print(safe_operation(10, 2))     # Works fine
print(safe_operation(10, 0))     # ZeroDivisionError
print(safe_operation(10, "2"))   # TypeError

## The else and finally Clauses

Python's `try`/`except` mechanism can be extended with two additional clauses:

- `else`: Executes if no exception was raised in the `try` block
- `finally`: Always executes, whether an exception was raised or not

In [ ]:
def read_and_process(filename):
    try:
        file = open(filename, 'r')
        content = file.read()
    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
        return None
    except IOError:
        print(f"Error: Could not read from '{filename}'.")
        return None
    else:
        # This block executes if no exceptions were raised in the try block
        print(f"Successfully read {len(content)} characters from '{filename}'.")
        return content.upper()  # Process the content
    finally:
        # This block always executes, regardless of whether an exception occurred
        print(f"Cleanup operations for '{filename}'.")
        # Close the file if it was opened
        if 'file' in locals() and not file.closed:
            file.close()
            print(f"File '{filename}' closed.")

# Create a test file
with open("test_file.txt", "w") as f:
    f.write("Hello, world!")

# Test the function
print("\nProcessing existing file:")
result = read_and_process("test_file.txt")
print(f"Result: {result}")

print("\nProcessing non-existent file:")
result = read_and_process("missing_file.txt")
print(f"Result: {result}")

### Common Use Cases for finally

The `finally` clause is particularly useful for resource cleanup operations, regardless of whether an error occurred. Besides file handling, it's commonly used for:

1. Database connections
2. Network sockets
3. Locks in concurrent programming
4. Any resource that needs to be released

In [ ]:
# Simulating a database connection
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name
        self.connected = False
        
    def connect(self):
        print(f"Connecting to database '{self.db_name}'...")
        self.connected = True
        print("Connected!")
        
    def execute(self, query):
        if not self.connected:
            raise ConnectionError("Not connected to database")
        if "error" in query.lower():
            raise ValueError("Query contains error keyword")
        print(f"Executing query: {query}")
        return ["result1", "result2"]  # Simulated results
        
    def close(self):
        if self.connected:
            print(f"Closing connection to '{self.db_name}'")
            self.connected = False
        
def query_database(db_name, query):
    db = DatabaseConnection(db_name)
    try:
        db.connect()
        results = db.execute(query)
        return results
    except ConnectionError as e:
        print(f"Connection error: {e}")
        return None
    except ValueError as e:
        print(f"Query error: {e}")
        return None
    finally:
        # Close the connection even if an error occurred
        db.close()

# Test with working query
print("Running working query:")
results = query_database("customer_data", "SELECT * FROM customers")
print(f"Results: {results}\n")

# Test with error query
print("Running query with error:")
results = query_database("customer_data", "SELECT * FROM customers WHERE error=True")
print(f"Results: {results}")

## Raising Exceptions

You can also raise exceptions explicitly using the `raise` statement. This is useful when you detect an error condition that Python doesn't automatically catch.

In [ ]:
def calculate_square_root(x):
    if not isinstance(x, (int, float)):
        raise TypeError(f"Expected a number, got {type(x).__name__}")
    if x < 0:
        raise ValueError("Cannot calculate square root of a negative number")
    return x ** 0.5

# Test with various inputs
try:
    print(f"Square root of 16: {calculate_square_root(16)}")
    print(f"Square root of -4: {calculate_square_root(-4)}")
except ValueError as e:
    print(f"Caught ValueError: {e}")

try:
    print(f"Square root of 'hello': {calculate_square_root('hello')}")
except TypeError as e:
    print(f"Caught TypeError: {e}")

### Re-raising Exceptions

Sometimes you want to catch an exception, do something with it, and then re-raise it (or a different exception). This is done using the `raise` statement without arguments within an `except` block:

In [ ]:
def process_positive_number(x):
    if x <= 0:
        raise ValueError("Expected a positive number")
    return x * 2

def safe_process(x):
    try:
        return process_positive_number(x)
    except ValueError as e:
        print(f"Warning: {e}")
        # Re-raise the exception after logging
        raise

# Test with positive number
print("Processing positive number:")
try:
    result = safe_process(5)
    print(f"Result: {result}")
except ValueError:
    print("Caught re-raised exception")

# Test with negative number
print("\nProcessing negative number:")
try:
    result = safe_process(-5)
    print(f"Result: {result}")
except ValueError:
    print("Caught re-raised exception")

## Creating Custom Exceptions

You can create your own exception classes by subclassing `Exception` or one of its subclasses. This is useful for creating application-specific exceptions that can be caught and handled separately.

In [ ]:
# Define custom exception classes
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the available balance"""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        self.deficit = amount - balance
        message = f"Cannot withdraw ${amount}. Balance is ${balance}, deficit is ${self.deficit}"
        super().__init__(message)

class AccountLockedError(Exception):
    """Raised when an operation is attempted on a locked account"""
    pass

# BankAccount class that uses the custom exceptions
class BankAccount:
    def __init__(self, account_number, initial_balance=0):
        self.account_number = account_number
        self.balance = initial_balance
        self.locked = False
    
    def deposit(self, amount):
        if self.locked:
            raise AccountLockedError(f"Account {self.account_number} is locked")
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount
        return self.balance
    
    def withdraw(self, amount):
        if self.locked:
            raise AccountLockedError(f"Account {self.account_number} is locked")
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        return self.balance
    
    def lock(self):
        self.locked = True
        
    def unlock(self):
        self.locked = False

# Test the BankAccount class with our custom exceptions
account = BankAccount("12345", 100)

# Normal operations
print(f"Initial balance: ${account.balance}")
account.deposit(50)
print(f"After deposit: ${account.balance}")
account.withdraw(30)
print(f"After withdrawal: ${account.balance}")

# InsufficientFundsError
try:
    account.withdraw(200)
except InsufficientFundsError as e:
    print(f"Error: {e}")
    print(f"You need ${e.deficit} more to make this withdrawal")

# Lock the account and try to withdraw
account.lock()
try:
    account.withdraw(10)
except AccountLockedError as e:
    print(f"Error: {e}")

# Unlock and continue
account.unlock()
account.withdraw(10)
print(f"Final balance: ${account.balance}")

## Best Practices for Error Handling

Here are some best practices for error handling in Python:

### 1. Be Specific with Your Exceptions

Avoid catching all exceptions with a bare `except:` clause as it can mask bugs. Instead, catch specific exceptions that you know how to handle.

In [ ]:
# Bad practice
def bad_practice(x):
    try:
        return 10 / x
    except:
        return 0  # Catches ALL exceptions, including KeyboardInterrupt, SystemExit, etc.

# Good practice
def good_practice(x):
    try:
        return 10 / x
    except ZeroDivisionError:
        return 0  # Only catches division by zero

print(f"Bad practice with x=0: {bad_practice(0)}")
print(f"Good practice with x=0: {good_practice(0)}")

# The bad practice would also catch a TypeError if we passed a string,
# which might hide bugs in your code
print(f"Bad practice with x='string': {bad_practice('string')}")
try:
    print(f"Good practice with x='string': {good_practice('string')}")
except TypeError as e:
    print(f"Good practice correctly allows TypeError to propagate: {e}")

### 2. Keep Try Blocks Small

Only put the code that might raise the exception in the try block, not the code that handles the result.

In [ ]:
# Bad practice - too much in the try block
def bad_file_read(filename):
    try:
        file = open(filename, 'r')
        content = file.read()
        file.close()
        lines = content.split('\n')
        return len(lines)
    except FileNotFoundError:
        return 0
    
# Good practice - minimal try block
def good_file_read(filename):
    try:
        file = open(filename, 'r')
    except FileNotFoundError:
        return 0
    
    try:
        content = file.read()
    finally:
        file.close()
    
    lines = content.split('\n')
    return len(lines)

### 3. Use Context Managers (with statement)

For resource management, use context managers with the `with` statement rather than try/finally blocks.

In [ ]:
# Without context manager
def count_lines_without_context_manager(filename):
    try:
        file = open(filename, 'r')
        try:
            return len(file.readlines())
        finally:
            file.close()
    except FileNotFoundError:
        return 0

# With context manager (cleaner and safer)
def count_lines_with_context_manager(filename):
    try:
        with open(filename, 'r') as file:
            return len(file.readlines())
    except FileNotFoundError:
        return 0

# Create a test file
with open("test_lines.txt", "w") as f:
    f.write("Line 1\nLine 2\nLine 3")

print(f"Without context manager: {count_lines_without_context_manager('test_lines.txt')}")
print(f"With context manager: {count_lines_with_context_manager('test_lines.txt')}")

### 4. Avoid Silent Failures

Don't suppress exceptions without at least logging them or taking appropriate action.

In [ ]:
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Bad practice - silently suppresses errors
def bad_silent_failure(data):
    try:
        return int(data)
    except ValueError:
        return 0  # Silently returns a default value

# Good practice - logs the error
def good_error_handling(data):
    try:
        return int(data)
    except ValueError:
        logging.warning(f"Could not convert '{data}' to an integer, using default value")
        return 0  # Returns default value but logs the issue

print(f"Bad practice with 'abc': {bad_silent_failure('abc')}")
print(f"Good practice with 'abc': {good_error_handling('abc')}")

### 5. Clean up Resources

Always clean up resources, even when exceptions occur, using `finally` blocks or context managers.

In [ ]:
# Simple context manager for demonstration
class TempFile:
    def __init__(self, filename):
        self.filename = filename
        self.file = None
        
    def __enter__(self):
        print(f"Creating temporary file {self.filename}")
        self.file = open(self.filename, 'w')
        return self.file
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        print(f"Temporary file {self.filename} closed")
        # Cleanup - delete the file
        import os
        try:
            os.remove(self.filename)
            print(f"Temporary file {self.filename} deleted")
        except FileNotFoundError:
            print(f"File {self.filename} already deleted or not found")
        # Return False to allow any exception to propagate
        return False

# Using the context manager
try:
    with TempFile('temp_demo.txt') as f:
        f.write("This is a temporary file")
        print("Writing to file...")
        # Simulate an error
        if True:  # Change to False to avoid the error
            raise RuntimeError("Simulated error")
        print("This line won't be reached if an error occurs")
except RuntimeError as e:
    print(f"Caught exception: {e}")
    
print("Program continues executing")

## Practical Error Handling Examples

Let's look at some practical examples of error handling in real-world scenarios.

### Example 1: Data Processing and Validation

In [ ]:
import csv
import io

# Sample CSV data (as a string for demonstration)
csv_data = """
id,name,age,email
1,John Doe,30,john@example.com
2,Jane Smith,25,jane@example.com
3,Bob Johnson,invalid,bob@example.com
4,Alice Brown,28,alice@example.com
5,Charlie Wilson,,charlie@example.com
"""

def process_user_data(csv_content):
    processed_users = []
    errors = []
    
    csv_file = io.StringIO(csv_content.strip())
    reader = csv.DictReader(csv_file)
    
    for i, row in enumerate(reader, start=1):
        try:
            # Validate required fields
            if not row['name']:
                raise ValueError("Name is required")
            
            # Process and validate age
            if row['age']:
                try:
                    age = int(row['age'])
                    if age < 0 or age > 120:
                        raise ValueError(f"Age {age} is out of realistic range (0-120)")
                    processed_age = age
                except ValueError:
                    errors.append(f"Row {i}: Invalid age '{row['age']}' for user {row['name']}")
                    processed_age = None
            else:
                processed_age = None
            
            # Validate email format (simple check)
            if '@' not in row['email']:
                raise ValueError(f"Invalid email format: {row['email']}")
            
            # Add processed user
            processed_users.append({
                'id': int(row['id']),
                'name': row['name'],
                'age': processed_age,
                'email': row['email']
            })
            
        except ValueError as e:
            errors.append(f"Row {i}: {e} for user {row.get('name', 'unknown')}")
        except Exception as e:
            errors.append(f"Row {i}: Unexpected error: {e}")
    
    return processed_users, errors

# Process the data
users, errors = process_user_data(csv_data)

print("Processed Users:")
for user in users:
    print(f"ID: {user['id']}, Name: {user['name']}, Age: {user['age']}, Email: {user['email']}")

print("\nErrors:")
for error in errors:
    print(error)

### Example 2: API Request Handling

In [ ]:
import urllib.request
import urllib.error
import json
import time

def fetch_user_data(user_id, max_retries=3, retry_delay=1):
    # Simulate different API endpoints
    urls = {
        1: "https://jsonplaceholder.typicode.com/users/1",  # Valid API
        2: "https://jsonplaceholder.typicode.com/nonexistent",  # 404 error
        3: "https://thisdomaindoesnotexist12345.com",  # DNS error
        4: "https://jsonplaceholder.typicode.com/users/4"  # Valid API
    }
    
    url = urls.get(user_id)
    if not url:
        raise ValueError(f"No URL configured for user_id {user_id}")
        
    retries = 0
    while retries < max_retries:
        try:
            print(f"Fetching data from {url}")
            with urllib.request.urlopen(url, timeout=5) as response:
                data = response.read().decode('utf-8')
                return json.loads(data)
                
        except urllib.error.HTTPError as e:
            print(f"HTTP Error: {e.code} - {e.reason}")
            if e.code == 404:
                # No point retrying for a 404
                raise ValueError(f"Resource not found at {url}")
            elif e.code >= 500:
                # Server error, might be worth retrying
                retries += 1
                if retries < max_retries:
                    print(f"Retrying in {retry_delay} seconds (attempt {retries}/{max_retries})")
                    time.sleep(retry_delay)
                else:
                    raise
            else:
                # Other HTTP errors, don't retry
                raise
                
        except urllib.error.URLError as e:
            print(f"URL Error: {e.reason}")
            retries += 1
            if retries < max_retries:
                print(f"Retrying in {retry_delay} seconds (attempt {retries}/{max_retries})")
                time.sleep(retry_delay)
            else:
                raise
                
        except json.JSONDecodeError as e:
            print(f"JSON Decode Error: {e}")
            # Invalid JSON is unlikely to be resolved by retrying
            raise ValueError(f"Invalid JSON response from {url}")
            
        except Exception as e:
            print(f"Unexpected error: {e}")
            raise
    
    raise TimeoutError(f"Maximum retries ({max_retries}) exceeded")

# Test with different user IDs
for user_id in [1, 2, 3, 4]:
    print(f"\nFetching user with ID {user_id}:")
    try:
        user_data = fetch_user_data(user_id)
        print(f"Successfully fetched user: {user_data['name']}")
    except Exception as e:
        print(f"Failed to fetch user with ID {user_id}: {e}")
    print("-" * 50)

## Exception Chaining

Sometimes, you want to catch one exception and raise a different one, while preserving the original exception information. This is called exception chaining and is done with the `from` keyword.

In [ ]:
class ConfigError(Exception):
    """Raised when there's an error in the configuration"""
    pass

def load_config(filename):
    try:
        with open(filename, 'r') as file:
            config_text = file.read()
        
        try:
            import json
            config = json.loads(config_text)
            return config
        except json.JSONDecodeError as e:
            # Chain the exception - preserves the original stack trace
            raise ConfigError(f"Invalid JSON in config file {filename}") from e
    except FileNotFoundError as e:
        raise ConfigError(f"Config file {filename} not found") from e

# First, create a valid config file
with open('valid_config.json', 'w') as f:
    f.write('{"debug": true, "log_level": "info"}')

# Then, create an invalid JSON config
with open('invalid_config.json', 'w') as f:
    f.write('{"debug": true, "log_level": "info" INVALID JSON}')

# Test with valid config
try:
    config = load_config('valid_config.json')
    print(f"Loaded valid config: {config}")
except ConfigError as e:
    print(f"Error loading config: {e}")

# Test with invalid config
try:
    config = load_config('invalid_config.json')
    print(f"Loaded config: {config}")
except ConfigError as e:
    print(f"Error loading config: {e}")
    print(f"Original exception: {e.__cause__}")

# Test with nonexistent config
try:
    config = load_config('nonexistent_config.json')
    print(f"Loaded config: {config}")
except ConfigError as e:
    print(f"Error loading config: {e}")
    print(f"Original exception: {e.__cause__}")

### Explicit vs Implicit Exception Chaining

Exception chaining can be explicit (using `from`) or implicit (when an exception is raised within an except block):

In [ ]:
def demo_explicit_chaining(value):
    try:
        result = 1 / value
    except ZeroDivisionError as e:
        # Explicit chaining with 'from'
        raise ValueError("Cannot process zero value") from e

def demo_implicit_chaining(value):
    try:
        result = 1 / value
    except ZeroDivisionError:
        # Implicit chaining - the ZeroDivisionError becomes __context__
        raise ValueError("Cannot process zero value")
        
def demo_suppressed_chaining(value):
    try:
        result = 1 / value
    except ZeroDivisionError:
        # Suppresses chaining with 'from None'
        raise ValueError("Cannot process zero value") from None

# Test explicit chaining
print("Explicit chaining:")
try:
    demo_explicit_chaining(0)
except ValueError as e:
    print(f"Error: {e}")
    print(f"__cause__: {e.__cause__}")
    print(f"__context__: {e.__context__}")

# Test implicit chaining
print("\nImplicit chaining:")
try:
    demo_implicit_chaining(0)
except ValueError as e:
    print(f"Error: {e}")
    print(f"__cause__: {e.__cause__}")
    print(f"__context__: {e.__context__}")

# Test suppressed chaining
print("\nSuppressed chaining:")
try:
    demo_suppressed_chaining(0)
except ValueError as e:
    print(f"Error: {e}")
    print(f"__cause__: {e.__cause__}")
    print(f"__context__: {e.__context__}")

## Summary

- Exception handling in Python is done with `try`/`except` blocks
- Be specific about which exceptions you catch
- Use `else` for code that should execute when no exceptions occur
- Use `finally` for cleanup code that should always execute
- Create custom exceptions by subclassing `Exception`
- Use exception chaining with `raise ... from ...` to preserve context
- Best practices include keeping try blocks small, avoiding silent failures, and proper resource cleanup

Effective error handling makes your code more robust, easier to maintain, and provides a better experience for users of your software.